# Task 3: Domain Generalization (PACS)

ERM (load Task 2 Source-only) → DAN-DG → SAM, plus λ_DG study and final Sketch eval.

Checkpoint selection uses **mean source-val macro-F1 only**. Sketch labels appear only in the final eval stage.

In [ ]:
import torch

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

REPO_DIR = Path("/content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA1")
os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

In [ ]:
%pip install -q -r requirements.txt gdown

## PACS + splits (reuse Task 2)

Skip download if `data/pacs/` already exists from Task 2.

In [ ]:
from pathlib import Path
import shutil
import zipfile
import urllib.request

pacs_root = Path("data/pacs")
marker = pacs_root / "photo"

def _count(d):
    return sum(1 for p in (pacs_root / d).rglob("*") if p.is_file())

if marker.is_dir() and _count("photo") > 0:
    print("PACS already present at", pacs_root.resolve())
else:
    pacs_root.parent.mkdir(parents=True, exist_ok=True)
    zip_path = Path("data/PACS.zip")
    if not zip_path.exists():
        url = "https://huggingface.co/datasets/Azeez577/PACS/resolve/main/PACS.zip"
        print("Downloading", url)
        urllib.request.urlretrieve(url, zip_path)
    extract_dir = Path("data/_pacs_extract")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    candidates = [c.parent for c in extract_dir.rglob("photo") if c.is_dir()]
    if not candidates:
        raise FileNotFoundError("Could not locate PACS photo/ after unzip")
    src = candidates[0]
    if pacs_root.exists():
        shutil.rmtree(pacs_root)
    shutil.move(str(src), str(pacs_root))
    print("Installed PACS ->", pacs_root.resolve())

for d in ["photo", "art_painting", "cartoon", "sketch"]:
    print(f"  {d}: {_count(d)} files")

In [ ]:
!python -m shared.prepare_pacs_splits
!python -m task3.scripts.run_task3 --stages check_erm

## Train main methods (DAN-DG λ=1, SAM ρ=0.05)

ERM is **not** retrained — it loads `task2/results/checkpoints/source_only_best.pt`.

In [ ]:
!python -m task3.scripts.run_task3 --stages train_main

## Controlled study: λ_DG ∈ {0.1, 1, 10}

Main comparison stays at λ_DG=1. Do not pick a winner from Sketch.

In [ ]:
!python -m task3.scripts.run_task3 --stages study_lambda

## Source-side diagnostics (no Sketch)

In [ ]:
!python -m task3.scripts.run_task3 --stages eval_source

## Final Sketch evaluation

Run only after all Task 3 decisions are fixed.

In [ ]:
!python -m task3.scripts.run_task3 --stages eval